# 第14章 - 通信与互联网技术（二）：互联网技术
# Chapter 14 - Communication & Internet Technologies (Part 2): Internet Technologies

---

## 🎯 学习目标 (Learning Objectives)

本节课对应 **Cambridge A-Level Computer Science (9618) Paper 3** 考纲要求：

| 考纲编号 | 内容 |
|---------|------|
| 14.6 | 理解 **DNS** 域名解析过程（recursive 和 iterative） |
| 14.7 | 理解 **HTTP/HTTPS** 协议的请求/响应结构 |
| 14.8 | 理解 **SSL/TLS** 加密握手过程 |
| 14.9 | 了解 Web 技术：**HTML, CSS, JavaScript** |
| 14.10 | 理解 **REST API** 和 **JSON** 数据格式 |
| 14.11 | 理解电子邮件协议：**SMTP, POP3, IMAP** |

---

> **前置知识**：你需要先完成 `01_网络协议栈.ipynb`，了解 TCP/IP 协议栈、端口号和 Socket 的基本概念。

In [ ]:
# ============================================================
# 环境配置 - 每次运行 notebook 前都需要先执行此 cell
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import json
import textwrap

# 中文字体配置
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("✅ 环境配置完成！")

---

## 1. DNS 域名系统 (Domain Name System)

### 1.1 什么是 DNS？

**DNS (Domain Name System)** 是互联网的 "电话簿"。它将人类可读的 **domain name（域名）** 转换为计算机使用的 **IP address（IP 地址）**。

```
www.google.com  →  DNS 解析  →  142.250.80.46
```

**为什么需要 DNS？**
- 人类更容易记忆域名（如 `www.google.com`）而不是数字（如 `142.250.80.46`）
- IP 地址可能会改变，但域名可以保持不变
- 一个域名可以对应多个 IP（负载均衡）

### 1.2 DNS 层次结构

DNS 采用 **hierarchical（分层）** 结构：

```
                        . (Root)
                       / | \
                     /   |   \
                  .com  .org  .cn   ← Top-Level Domain (TLD)
                 / | \
              /    |    \
         google  amazon  github    ← Second-Level Domain (SLD)
           |       |       |
          www     www     www      ← Subdomain
```

#### DNS 服务器类型：

| 服务器类型 | 英文名 | 功能 |
|-----------|--------|------|
| **根域名服务器** | Root Name Server | 知道所有 TLD 服务器的地址（全球共 13 组） |
| **顶级域名服务器** | TLD Name Server | 管理 `.com`, `.org`, `.cn` 等顶级域 |
| **权威域名服务器** | Authoritative Name Server | 存储特定域名的实际 IP 记录 |
| **递归解析器** | Recursive Resolver | 代替客户端完成整个查询过程（通常由 ISP 提供） |

### 1.3 DNS 解析过程

DNS 解析有两种模式：**Recursive（递归）** 和 **Iterative（迭代）**。

#### 递归查询 (Recursive Query)

客户端只需要向**递归解析器**发送一次请求，解析器负责完成所有后续查询：

```
Client → Recursive Resolver: "www.example.com 的 IP 是什么？"
         Resolver 自行完成以下查询：
         Resolver → Root Server: ".com 的 TLD 服务器在哪？"
         Resolver → TLD Server:  "example.com 的权威服务器在哪？"
         Resolver → Auth Server: "www.example.com 的 IP 是什么？"
Recursive Resolver → Client: "IP 是 93.184.216.34"
```

#### 迭代查询 (Iterative Query)

每个服务器不直接回答，而是告诉查询者 "下一步该问谁"：

```
Resolver → Root Server: "www.example.com?"
Root Server → Resolver: "我不知道，去问 .com TLD 服务器"

Resolver → TLD Server: "www.example.com?"
TLD Server → Resolver: "我不知道，去问 example.com 的权威服务器"

Resolver → Authoritative Server: "www.example.com?"
Authoritative Server → Resolver: "IP 是 93.184.216.34"
```

#### DNS Caching（缓存）

为了提高效率，DNS 结果会被 **缓存 (cache)**：
- **浏览器缓存** → 检查浏览器之前是否查过
- **操作系统缓存** → 检查本机 DNS 缓存
- **递归解析器缓存** → ISP 的 DNS 服务器缓存

每条 DNS 记录都有一个 **TTL (Time To Live)**，缓存过期后需重新查询。

In [ ]:
# ============================================================
# 可视化：DNS 解析过程 (DNS Resolution Process)
# ============================================================

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('DNS Resolution Process（DNS 解析过程）', fontsize=18, fontweight='bold', pad=20)

# 定义各个角色的位置
actors = {
    'client': (1.5, 6, 'Client\n客户端', '#AED6F1'),
    'resolver': (5.5, 6, 'Recursive\nResolver\n递归解析器', '#F9E79F'),
    'root': (10, 10, 'Root\nServer\n根服务器', '#F5B7B1'),
    'tld': (13, 7, 'TLD\nServer\n(.com)', '#D5F5E3'),
    'auth': (10, 3, 'Authoritative\nServer\n权威服务器', '#D2B4DE'),
}

# 画角色框
for key, (x, y, label, color) in actors.items():
    rect = FancyBboxPatch((x - 1.2, y - 0.8), 2.4, 1.6,
                          boxstyle="round,pad=0.15", facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold')

# 画查询箭头和编号
steps = [
    (1.5, 5.2, 5.5, 5.2, '1. www.example.com?', '#E74C3C', 0.15),
    (5.5, 6.8, 10, 9.2, '2. .com TLD?', '#3498DB', 0.1),
    (10, 9.2, 5.5, 6.8, '3. TLD Server IP', '#3498DB', -0.1),
    (5.5, 6.5, 13, 6.2, '4. example.com?', '#27AE60', 0.1),
    (13, 6.2, 5.5, 6.5, '5. Auth Server IP', '#27AE60', -0.1),
    (5.5, 5.2, 10, 3.8, '6. www.example.com?', '#8E44AD', 0.1),
    (10, 3.8, 5.5, 5.2, '7. 93.184.216.34', '#8E44AD', -0.1),
    (5.5, 5.2, 1.5, 5.2, '8. 93.184.216.34', '#E74C3C', -0.15),
]

for x1, y1, x2, y2, label, color, offset in steps:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    mid_x = (x1 + x2) / 2
    mid_y = (y1 + y2) / 2 + offset + 0.3
    ax.text(mid_x, mid_y, label, ha='center', va='center', fontsize=8, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=color, alpha=0.8))

# 底部说明
ax.text(8, 0.8, 'DNS 使用 UDP port 53 进行通信（查询结果较大时使用 TCP）',
        ha='center', va='center', fontsize=11, style='italic', color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 演示：模拟 DNS 解析过程
# ============================================================

class DNSSimulator:
    """模拟 DNS 解析过程的教学演示"""
    
    def __init__(self):
        # 模拟的 DNS 数据库
        self.root_servers = {
            'com': 'TLD-Server-COM (198.41.0.4)',
            'org': 'TLD-Server-ORG (199.9.14.201)',
            'cn': 'TLD-Server-CN (192.33.4.12)',
        }
        self.tld_servers = {
            'google.com': 'Auth-Server-Google (216.239.32.10)',
            'example.com': 'Auth-Server-Example (199.43.135.53)',
            'github.com': 'Auth-Server-GitHub (140.82.121.4)',
        }
        self.auth_servers = {
            'www.google.com': '142.250.80.46',
            'www.example.com': '93.184.216.34',
            'www.github.com': '140.82.121.3',
            'mail.google.com': '142.250.80.17',
        }
        self.cache = {}  # DNS 缓存
    
    def resolve(self, domain):
        """模拟迭代 DNS 查询"""
        print(f"\n{'='*60}")
        print(f"开始解析域名: {domain}")
        print(f"{'='*60}")
        
        # Step 0: 检查缓存
        if domain in self.cache:
            print(f"\n[Cache HIT] 缓存命中！")
            print(f"  {domain} → {self.cache[domain]}")
            return self.cache[domain]
        print(f"\n[Cache MISS] 缓存未命中，开始查询...")
        
        # 解析域名结构
        parts = domain.split('.')
        tld = parts[-1]  # 例如 'com'
        sld = '.'.join(parts[-2:])  # 例如 'google.com'
        
        # Step 1: 查询根服务器
        print(f"\nStep 1: 查询 Root Server（根服务器）")
        print(f"  问: \".{tld} 的 TLD 服务器在哪？\"")
        if tld in self.root_servers:
            tld_server = self.root_servers[tld]
            print(f"  答: \"{tld_server}\"")
        else:
            print(f"  错误: 未知的 TLD '.{tld}'")
            return None
        
        # Step 2: 查询 TLD 服务器
        print(f"\nStep 2: 查询 TLD Server（{tld_server}）")
        print(f"  问: \"{sld} 的权威服务器在哪？\"")
        if sld in self.tld_servers:
            auth_server = self.tld_servers[sld]
            print(f"  答: \"{auth_server}\"")
        else:
            print(f"  错误: 未知的域名 '{sld}'")
            return None
        
        # Step 3: 查询权威服务器
        print(f"\nStep 3: 查询 Authoritative Server（{auth_server}）")
        print(f"  问: \"{domain} 的 IP 地址是什么？\"")
        if domain in self.auth_servers:
            ip = self.auth_servers[domain]
            print(f"  答: \"{ip}\"")
        else:
            print(f"  错误: 未知的主机名 '{domain}'")
            return None
        
        # 缓存结果
        self.cache[domain] = ip
        print(f"\n[Cache STORE] 结果已缓存")
        print(f"\n✅ 解析完成: {domain} → {ip}")
        return ip

# 运行模拟
dns = DNSSimulator()

# 第一次查询（无缓存）
dns.resolve('www.google.com')

# 第二次查询同一域名（有缓存）
dns.resolve('www.google.com')

# 查询另一个域名
dns.resolve('www.example.com')

---

## 2. HTTP/HTTPS 协议

### 2.1 HTTP (HyperText Transfer Protocol)

**HTTP** 是 Web 浏览器和 Web 服务器之间通信的协议。它是一个 **stateless（无状态）** 的 **request-response（请求-响应）** 协议。

- **端口号**：80（HTTP）/ 443（HTTPS）
- **传输层协议**：TCP
- **无状态**：每个请求都是独立的，服务器不记住之前的请求

### 2.2 HTTP Request（请求）结构

```
GET /index.html HTTP/1.1          ← Request Line（请求行）
Host: www.example.com             ← Headers（请求头）
User-Agent: Mozilla/5.0           ← 
Accept: text/html                 ←
Accept-Language: zh-CN            ←
Connection: keep-alive            ←
                                  ← 空行（分隔头部和正文）
[Request Body]                    ← Body（请求正文，GET 请求通常为空）
```

#### HTTP 请求方法（考试重点）：

| 方法 | 用途 | 是否有 Body | 幂等性 |
|------|------|-----------|--------|
| **GET** | 获取资源 | 无 | 是 |
| **POST** | 提交数据（创建） | 有 | 否 |
| **PUT** | 更新资源（完整替换） | 有 | 是 |
| **DELETE** | 删除资源 | 无 | 是 |
| **PATCH** | 更新资源（部分修改） | 有 | 否 |
| **HEAD** | 类似 GET，但只返回头部 | 无 | 是 |

### 2.3 HTTP Response（响应）结构

```
HTTP/1.1 200 OK                   ← Status Line（状态行）
Content-Type: text/html           ← Headers（响应头）
Content-Length: 1234              ←
Date: Mon, 15 Apr 2026 08:00:00  ←
Server: Apache/2.4               ←
                                  ← 空行
<html>                            ← Body（响应正文）
  <body>Hello World</body>
</html>
```

#### HTTP 状态码（考试常考）：

| 状态码 | 含义 | 类别 |
|--------|------|------|
| **200** | OK - 请求成功 | 2xx Success |
| **201** | Created - 资源创建成功 | 2xx Success |
| **301** | Moved Permanently - 永久重定向 | 3xx Redirection |
| **302** | Found - 临时重定向 | 3xx Redirection |
| **400** | Bad Request - 请求格式错误 | 4xx Client Error |
| **401** | Unauthorized - 未授权 | 4xx Client Error |
| **403** | Forbidden - 禁止访问 | 4xx Client Error |
| **404** | Not Found - 资源未找到 | 4xx Client Error |
| **500** | Internal Server Error - 服务器内部错误 | 5xx Server Error |

In [ ]:
# ============================================================
# 演示：模拟 HTTP 请求和响应
# ============================================================

def simulate_http_request(method, path, host, headers=None, body=None):
    """模拟构造一个 HTTP 请求并展示其结构"""
    print(f"{'='*65}")
    print(f"HTTP Request（HTTP 请求）")
    print(f"{'='*65}")
    
    # 构造请求行
    request_line = f"{method} {path} HTTP/1.1"
    
    # 默认请求头
    default_headers = {
        'Host': host,
        'User-Agent': 'Mozilla/5.0 (Educational Demo)',
        'Accept': 'text/html, application/json',
        'Accept-Language': 'zh-CN, en-US',
        'Accept-Encoding': 'gzip, deflate',
        'Connection': 'keep-alive',
    }
    if headers:
        default_headers.update(headers)
    
    # 打印请求
    print(f"\n{request_line}")
    for key, value in default_headers.items():
        print(f"{key}: {value}")
    print()  # 空行
    if body:
        print(body)
    else:
        print("(no body / 无请求正文)")
    
    return request_line, default_headers

def simulate_http_response(status_code, status_text, content_type, body):
    """模拟构造一个 HTTP 响应"""
    print(f"\n{'='*65}")
    print(f"HTTP Response（HTTP 响应）")
    print(f"{'='*65}")
    
    status_line = f"HTTP/1.1 {status_code} {status_text}"
    response_headers = {
        'Content-Type': content_type,
        'Content-Length': str(len(body.encode('utf-8'))),
        'Date': 'Tue, 15 Apr 2026 08:00:00 GMT',
        'Server': 'Apache/2.4.41 (Ubuntu)',
        'Cache-Control': 'max-age=3600',
    }
    
    print(f"\n{status_line}")
    for key, value in response_headers.items():
        print(f"{key}: {value}")
    print()  # 空行
    print(body)

# === 示例 1: GET 请求 ===
print("\n" + "#" * 65)
print("示例 1: GET 请求 - 获取网页")
print("#" * 65)
simulate_http_request('GET', '/index.html', 'www.example.com')
simulate_http_response(200, 'OK', 'text/html; charset=utf-8',
    '<html>\n<head><title>Example</title></head>\n<body><h1>Hello World!</h1></body>\n</html>')

# === 示例 2: POST 请求 ===
print("\n\n" + "#" * 65)
print("示例 2: POST 请求 - 提交表单数据")
print("#" * 65)
simulate_http_request('POST', '/api/login', 'www.example.com',
    headers={'Content-Type': 'application/json'},
    body='{"username": "student", "password": "abc123"}')
simulate_http_response(200, 'OK', 'application/json',
    '{"status": "success", "token": "eyJhbGciOiJIUz..."}')

In [ ]:
# ============================================================
# 可视化：HTTP 请求-响应过程
# ============================================================

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('HTTP Request-Response Cycle（HTTP 请求-响应周期）',
             fontsize=16, fontweight='bold', pad=15)

# Client
rect_c = FancyBboxPatch((0.5, 5), 3, 2, boxstyle="round,pad=0.2",
                         facecolor='#AED6F1', edgecolor='black', linewidth=2)
ax.add_patch(rect_c)
ax.text(2, 6, 'Client\n(Browser)\n浏览器', ha='center', va='center', fontsize=12, fontweight='bold')

# Server
rect_s = FancyBboxPatch((10.5, 5), 3, 2, boxstyle="round,pad=0.2",
                         facecolor='#ABEBC6', edgecolor='black', linewidth=2)
ax.add_patch(rect_s)
ax.text(12, 6, 'Server\n(Web Server)\nWeb 服务器', ha='center', va='center', fontsize=12, fontweight='bold')

# Step 1: DNS
ax.annotate('', xy=(7, 10.5), xytext=(2, 8.5),
            arrowprops=dict(arrowstyle='->', color='#9B59B6', lw=2, connectionstyle='arc3,rad=0.1'))
ax.text(3.5, 10, '1. DNS Lookup\n域名解析', ha='center', va='center', fontsize=10, color='#9B59B6', fontweight='bold')
rect_dns = FancyBboxPatch((5.8, 10), 2.4, 1, boxstyle="round,pad=0.15",
                           facecolor='#F5B7B1', edgecolor='black')
ax.add_patch(rect_dns)
ax.text(7, 10.5, 'DNS Server', ha='center', va='center', fontsize=10)

# Step 2: TCP Handshake
ax.annotate('', xy=(10.5, 6.8), xytext=(3.5, 6.8),
            arrowprops=dict(arrowstyle='<->', color='#E67E22', lw=2))
ax.text(7, 7.2, '2. TCP Three-Way Handshake\nTCP 三次握手', ha='center', va='center', fontsize=10,
        color='#E67E22', fontweight='bold')

# Step 3: HTTP Request
ax.annotate('', xy=(10.5, 6), xytext=(3.5, 6),
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2.5))
ax.text(7, 6.3, '3. HTTP Request (GET /index.html)', ha='center', fontsize=10,
        color='#E74C3C', fontweight='bold')

# Step 4: HTTP Response
ax.annotate('', xy=(3.5, 5.2), xytext=(10.5, 5.2),
            arrowprops=dict(arrowstyle='->', color='#27AE60', lw=2.5))
ax.text(7, 4.6, '4. HTTP Response (200 OK + HTML)', ha='center', fontsize=10,
        color='#27AE60', fontweight='bold')

# Step 5: Render
ax.text(2, 3.5, '5. Browser renders\nthe HTML page\n浏览器渲染页面', ha='center', va='center', fontsize=10,
        color='#2980B9', fontweight='bold')

# 底部说明
status_codes = '2xx = Success | 3xx = Redirect | 4xx = Client Error | 5xx = Server Error'
ax.text(7, 1.0, f'HTTP Status Codes: {status_codes}',
        ha='center', va='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#F8F9FA', edgecolor='gray'))

plt.tight_layout()
plt.show()

---

## 3. SSL/TLS 与 HTTPS

### 3.1 HTTPS = HTTP + SSL/TLS

**HTTPS (HTTP Secure)** 在 HTTP 的基础上添加了 **SSL/TLS** 加密层，确保数据在传输过程中的安全性。

| 特征 | HTTP | HTTPS |
|------|------|-------|
| 端口 | 80 | **443** |
| 加密 | 无（明文传输） | **SSL/TLS 加密** |
| 安全性 | 不安全 | 安全 |
| URL 前缀 | `http://` | `https://` |
| 证书 | 不需要 | 需要 **digital certificate** |
| 速度 | 较快 | 稍慢（加密开销） |

### 3.2 SSL/TLS Handshake（握手过程）

**TLS (Transport Layer Security)** 是 SSL 的继任者。TLS 握手在 TCP 三次握手之后进行：

```
Client                                    Server
  |                                         |
  |--- 1. Client Hello ------------------->|  (支持的 TLS 版本、加密算法列表、随机数)
  |                                         |
  |<-- 2. Server Hello --------------------|  (选择的 TLS 版本、加密算法、随机数)
  |<-- 3. Server Certificate --------------|  (服务器的数字证书，含公钥)
  |<-- 4. Server Hello Done --------------- |  
  |                                         |
  |--- 5. Client Key Exchange ------------>|  (用服务器公钥加密的 pre-master secret)
  |--- 6. Change Cipher Spec ------------->|  (通知切换到加密通信)
  |--- 7. Finished ----------------------->|  (加密验证消息)
  |                                         |
  |<-- 8. Change Cipher Spec --------------|  (服务器也切换到加密)
  |<-- 9. Finished ------------------------|  (加密验证消息)
  |                                         |
  |========= 加密通信开始 ==================|
```

### 3.3 关键概念

- **Digital Certificate（数字证书）**：由 **CA (Certificate Authority)** 签发，证明服务器的身份
- **Public Key（公钥）**：包含在证书中，用于加密
- **Private Key（私钥）**：服务器保密，用于解密
- **Symmetric Encryption（对称加密）**：实际数据传输使用（速度快）
- **Asymmetric Encryption（非对称加密）**：握手阶段使用（安全交换对称密钥）

> **考试重点**：TLS 握手使用**非对称加密**交换密钥，之后的数据传输使用**对称加密**，这样兼顾了安全性和速度。

In [ ]:
# ============================================================
# 可视化：TLS Handshake 过程
# ============================================================

fig, ax = plt.subplots(figsize=(14, 11))
ax.set_xlim(0, 14)
ax.set_ylim(0, 13)
ax.axis('off')
ax.set_title('TLS Handshake Process（TLS 握手过程）', fontsize=18, fontweight='bold', pad=15)

# Client 和 Server
ax.text(2.5, 12.3, 'Client (Browser)', ha='center', fontsize=14, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#AED6F1', edgecolor='black', linewidth=2))
ax.text(11.5, 12.3, 'Server', ha='center', va='center', fontsize=14, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#ABEBC6', edgecolor='black', linewidth=2))

# 时间线
ax.plot([2.5, 2.5], [11.5, 1.5], 'b-', linewidth=2, alpha=0.5)
ax.plot([11.5, 11.5], [11.5, 1.5], 'g-', linewidth=2, alpha=0.5)

# 握手步骤
tls_steps = [
    (10.8, 2.7, 11.3, 'Client Hello', '#E74C3C',
     'TLS version, cipher suites,\nclient random'),
    (9.2, 11.3, 2.7, 'Server Hello', '#3498DB',
     'Chosen cipher suite,\nserver random'),
    (8.0, 11.3, 2.7, 'Certificate', '#8E44AD',
     'Server public key\n(digital certificate)'),
    (6.8, 11.3, 2.7, 'Server Hello Done', '#7F8C8D', ''),
    (5.6, 2.7, 11.3, 'Client Key Exchange', '#E67E22',
     'Pre-master secret\n(encrypted with server public key)'),
    (4.4, 2.7, 11.3, 'Change Cipher Spec', '#27AE60',
     'Switch to encrypted mode'),
    (3.2, 11.3, 2.7, 'Change Cipher Spec', '#27AE60',
     'Server also switches'),
]

for y, x_from, x_to, label, color, detail in tls_steps:
    ax.annotate('', xy=(x_to, y), xytext=(x_from, y),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    mid_x = (x_from + x_to) / 2
    ax.text(mid_x, y + 0.35, label, ha='center', va='center', fontsize=10, fontweight='bold', color=color)
    if detail:
        ax.text(mid_x, y - 0.3, detail, ha='center', va='center', fontsize=7, color='gray', style='italic')

# 加密通信区域
rect_enc = plt.Rectangle((1, 1.5), 12, 1, facecolor='#D5F5E3', edgecolor='#27AE60',
                          linewidth=2, linestyle='--', alpha=0.5)
ax.add_patch(rect_enc)
ax.text(7, 2.0, 'Encrypted Communication（加密通信）\nAll data now encrypted with symmetric key',
        ha='center', va='center', fontsize=11, fontweight='bold', color='#27AE60')

plt.tight_layout()
plt.show()

---

## 4. Web 技术概览 (Web Technologies Overview)

现代 Web 由三大核心技术构成：

### 4.1 HTML (HyperText Markup Language)

**HTML** 定义网页的 **结构和内容 (structure & content)**。

```html
<!DOCTYPE html>
<html>
<head>
    <title>My Page</title>
</head>
<body>
    <h1>Hello World</h1>
    <p>This is a paragraph.</p>
    <a href="https://example.com">Link</a>
    <img src="photo.jpg" alt="A photo">
</body>
</html>
```

**关键概念**：
- 使用 **tags（标签）** 标记内容（如 `<h1>`, `<p>`, `<a>`）
- **Hyperlinks（超链接）**：`<a href="...">` 连接不同的网页
- 浏览器解析 HTML 后渲染（render）为可视化页面

### 4.2 CSS (Cascading Style Sheets)

**CSS** 定义网页的 **样式和外观 (style & appearance)**。

```css
body {
    background-color: #f0f0f0;
    font-family: Arial, sans-serif;
}
h1 {
    color: blue;
    font-size: 24px;
}
```

**关键概念**：
- **分离关注点**：HTML 管内容，CSS 管外观
- 使用 **selectors（选择器）** 选中要样式化的元素
- 可以控制颜色、字体、布局、动画等

### 4.3 JavaScript

**JavaScript** 定义网页的 **行为和交互 (behavior & interactivity)**。

```javascript
document.getElementById('myButton').addEventListener('click', function() {
    alert('Button clicked!');
});
```

**关键概念**：
- **Client-side scripting（客户端脚本）**：在浏览器中执行
- 可以动态修改 HTML/CSS
- 处理用户交互（点击、输入等）
- 通过 **AJAX** 与服务器异步通信

### 4.4 三者的关系

| 技术 | 作用 | 类比 |
|------|------|------|
| **HTML** | 结构 (Structure) | 房子的框架和墙壁 |
| **CSS** | 样式 (Style) | 房子的装修和油漆 |
| **JavaScript** | 行为 (Behavior) | 房子的电路和智能设备 |

In [ ]:
# ============================================================
# 可视化：Web 三大技术
# ============================================================

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Web Technologies: HTML + CSS + JavaScript',
             fontsize=16, fontweight='bold', pad=15)

# HTML 层
rect1 = FancyBboxPatch((1, 0.5), 10, 2, boxstyle="round,pad=0.2",
                         facecolor='#FFB3B3', edgecolor='#E74C3C', linewidth=2)
ax.add_patch(rect1)
ax.text(6, 1.5, 'HTML - Structure（结构）\n定义网页的内容和结构\n标签: <html>, <head>, <body>, <h1>, <p>, <a>, <img>',
        ha='center', va='center', fontsize=11, fontweight='bold')

# CSS 层
rect2 = FancyBboxPatch((1, 2.8), 10, 2, boxstyle="round,pad=0.2",
                         facecolor='#B3D9FF', edgecolor='#3498DB', linewidth=2)
ax.add_patch(rect2)
ax.text(6, 3.8, 'CSS - Style（样式）\n定义网页的外观和布局\n属性: color, font-size, background, margin, padding',
        ha='center', va='center', fontsize=11, fontweight='bold')

# JavaScript 层
rect3 = FancyBboxPatch((1, 5.1), 10, 2, boxstyle="round,pad=0.2",
                         facecolor='#FFFFB3', edgecolor='#F1C40F', linewidth=2)
ax.add_patch(rect3)
ax.text(6, 6.1, 'JavaScript - Behavior（行为）\n定义网页的交互和动态功能\n功能: DOM manipulation, event handling, AJAX',
        ha='center', va='center', fontsize=11, fontweight='bold')

# 箭头
ax.annotate('', xy=(0.5, 6.1), xytext=(0.5, 1.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(0.3, 3.8, 'Layers', ha='center', va='center', fontsize=10, rotation=90, color='gray')

plt.tight_layout()
plt.show()

---

## 5. REST API 和 JSON

### 5.1 什么是 API？

**API (Application Programming Interface)** 是不同软件组件之间通信的接口。**Web API** 允许通过 HTTP 请求访问服务器上的数据和功能。

### 5.2 REST (Representational State Transfer)

**REST** 是一种设计 Web API 的架构风格。符合 REST 原则的 API 叫做 **RESTful API**。

#### REST 的核心原则：

1. **Client-Server（客户端-服务器）**：前后端分离
2. **Stateless（无状态）**：每个请求包含所有必要信息
3. **Uniform Interface（统一接口）**：使用标准 HTTP 方法
4. **Resource-based（基于资源）**：用 URL 标识资源

#### RESTful API 设计示例：

| HTTP 方法 | URL | 功能 | 描述 |
|-----------|-----|------|------|
| GET | `/api/students` | 获取所有学生 | Read (读取) |
| GET | `/api/students/123` | 获取学生 #123 | Read (读取) |
| POST | `/api/students` | 创建新学生 | Create (创建) |
| PUT | `/api/students/123` | 更新学生 #123 | Update (更新) |
| DELETE | `/api/students/123` | 删除学生 #123 | Delete (删除) |

这就是常说的 **CRUD**：**C**reate, **R**ead, **U**pdate, **D**elete。

### 5.3 JSON (JavaScript Object Notation)

**JSON** 是 Web API 中最常用的数据交换格式。它是轻量级、易于阅读的文本格式。

```json
{
    "name": "张三",
    "age": 17,
    "subjects": ["Computer Science", "Mathematics", "Physics"],
    "enrolled": true,
    "address": {
        "city": "Beijing",
        "country": "China"
    }
}
```

**JSON 数据类型**：
- **String（字符串）**：`"hello"`
- **Number（数字）**：`42`, `3.14`
- **Boolean（布尔值）**：`true`, `false`
- **Array（数组）**：`[1, 2, 3]`
- **Object（对象）**：`{"key": "value"}`
- **null**：`null`

In [ ]:
# ============================================================
# 演示：JSON 数据操作
# ============================================================
import json

print("=" * 60)
print("JSON 数据操作演示")
print("=" * 60)

# 创建一个 JSON 数据结构
student_data = {
    "id": 1001,
    "name": "张三",
    "age": 17,
    "grade": "A2",
    "subjects": [
        {"name": "Computer Science", "code": "9618", "score": 85},
        {"name": "Mathematics", "code": "9709", "score": 92},
        {"name": "Physics", "code": "9702", "score": 78}
    ],
    "contact": {
        "email": "zhangsan@example.com",
        "phone": "+86-138-xxxx-xxxx"
    },
    "enrolled": True
}

# Python dict → JSON string
json_string = json.dumps(student_data, ensure_ascii=False, indent=2)
print("\n1. Python dict → JSON string (序列化):")
print(json_string)

# JSON string → Python dict
parsed_data = json.loads(json_string)
print("\n2. JSON string → Python dict (反序列化):")
print(f"   学生姓名: {parsed_data['name']}")
print(f"   年龄: {parsed_data['age']}")
print(f"   科目数量: {len(parsed_data['subjects'])}")

# 访问嵌套数据
print("\n3. 访问嵌套数据:")
for subject in parsed_data['subjects']:
    print(f"   {subject['name']} ({subject['code']}): {subject['score']} 分")

print(f"\n4. Email: {parsed_data['contact']['email']}")

In [ ]:
# ============================================================
# 演示：模拟 RESTful API 交互
# ============================================================

class SimpleRESTAPI:
    """模拟一个简单的 RESTful API（学生管理系统）"""
    
    def __init__(self):
        self.students = {
            1: {"id": 1, "name": "张三", "grade": "A2", "score": 85},
            2: {"id": 2, "name": "李四", "grade": "AS", "score": 72},
            3: {"id": 3, "name": "王五", "grade": "A2", "score": 91},
        }
        self.next_id = 4
    
    def handle_request(self, method, path, body=None):
        """处理 HTTP 请求"""
        print(f"\n{'─'*50}")
        print(f"→ {method} {path}")
        if body:
            print(f"  Body: {json.dumps(body, ensure_ascii=False)}")
        
        # 路由解析
        parts = path.strip('/').split('/')
        
        if method == 'GET' and path == '/api/students':
            # GET 所有学生
            response = list(self.students.values())
            return self._response(200, 'OK', response)
        
        elif method == 'GET' and len(parts) == 3:
            # GET 单个学生
            student_id = int(parts[2])
            if student_id in self.students:
                return self._response(200, 'OK', self.students[student_id])
            return self._response(404, 'Not Found', {"error": "Student not found"})
        
        elif method == 'POST' and path == '/api/students':
            # 创建新学生
            body['id'] = self.next_id
            self.students[self.next_id] = body
            self.next_id += 1
            return self._response(201, 'Created', body)
        
        elif method == 'DELETE' and len(parts) == 3:
            # 删除学生
            student_id = int(parts[2])
            if student_id in self.students:
                deleted = self.students.pop(student_id)
                return self._response(200, 'OK', {"message": f"Deleted student {deleted['name']}"})
            return self._response(404, 'Not Found', {"error": "Student not found"})
        
        return self._response(400, 'Bad Request', {"error": "Invalid request"})
    
    def _response(self, status, text, data):
        """格式化响应"""
        print(f"← {status} {text}")
        formatted = json.dumps(data, ensure_ascii=False, indent=2)
        print(f"  Response: {formatted}")
        return status, data

# 运行 API 演示
print("=" * 55)
print("RESTful API 模拟演示 - 学生管理系统")
print("=" * 55)

api = SimpleRESTAPI()

# GET 所有学生
api.handle_request('GET', '/api/students')

# GET 单个学生
api.handle_request('GET', '/api/students/1')

# POST 创建新学生
api.handle_request('POST', '/api/students',
                   {"name": "赵六", "grade": "AS", "score": 88})

# GET 不存在的学生 → 404
api.handle_request('GET', '/api/students/99')

# DELETE 学生
api.handle_request('DELETE', '/api/students/2')

# 查看更新后的列表
api.handle_request('GET', '/api/students')

---

## 6. 电子邮件协议 (Email Protocols)

电子邮件系统使用三种主要协议：

### 6.1 SMTP (Simple Mail Transfer Protocol)

- **用途**：**发送** 电子邮件
- **端口**：25（未加密）/ 587（TLS 加密）
- **方向**：客户端 → 服务器，或 服务器 → 服务器
- **协议类型**：TCP（需要可靠传输）

SMTP 只负责 **发送**，不负责接收。就像邮局只负责揽收和投递。

### 6.2 POP3 (Post Office Protocol version 3)

- **用途**：从服务器 **下载** 电子邮件
- **端口**：110（未加密）/ 995（SSL 加密）
- **工作方式**：下载邮件后通常从服务器**删除**
- **特点**：适合在**单一设备**上阅读邮件

### 6.3 IMAP (Internet Message Access Protocol)

- **用途**：从服务器 **同步** 电子邮件
- **端口**：143（未加密）/ 993（SSL 加密）
- **工作方式**：邮件保存在服务器上，客户端与服务器**同步**
- **特点**：适合在**多设备**上阅读邮件

### 6.4 POP3 vs IMAP 比较

| 特征 | POP3 | IMAP |
|------|------|------|
| 邮件存储位置 | **下载到本地** | **保留在服务器** |
| 多设备同步 | 不支持 | **支持** |
| 离线阅读 | **支持**（已下载） | 部分支持 |
| 服务器空间 | 节省 | 需要更多空间 |
| 网络带宽 | 首次下载后低 | 每次都需要连接 |
| 适用场景 | 单一设备 | **多设备** |
| 端口 | 110 / 995(SSL) | 143 / 993(SSL) |

In [ ]:
# ============================================================
# 可视化：电子邮件发送和接收流程
# ============================================================

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Email System: Sending & Receiving（电子邮件系统流程）',
             fontsize=16, fontweight='bold', pad=15)

# 定义元素
elements = [
    (1.5, 5, 2.2, 1.6, 'Sender\n发件人\n(Alice)', '#AED6F1'),
    (5.5, 5, 2.2, 1.6, 'Sender\'s\nMail Server\n发件服务器', '#F9E79F'),
    (10, 5, 2.2, 1.6, 'Receiver\'s\nMail Server\n收件服务器', '#F5B7B1'),
    (14, 5, 1.8, 1.6, 'Receiver\n收件人\n(Bob)', '#ABEBC6'),
]

for x, y, w, h, label, color in elements:
    rect = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle="round,pad=0.15", facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold')

# 箭头和标签
arrows = [
    (2.6, 5.8, 4.4, 5.8, 'SMTP\n(Port 587)', '#E74C3C'),
    (6.6, 5.8, 8.9, 5.8, 'SMTP\n(Port 25)', '#E74C3C'),
    (11.1, 4.5, 13.1, 4.5, 'POP3 (110)\nor IMAP (143)', '#3498DB'),
]

for x1, y1, x2, y2, label, color in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    mid_x = (x1 + x2) / 2
    mid_y = (y1 + y2) / 2
    offset = 0.55 if y1 > 5 else -0.55
    ax.text(mid_x, mid_y + offset, label, ha='center', va='center', fontsize=10,
            fontweight='bold', color=color)

# 说明框
notes = [
    (4, 2.5, 'SMTP: 只负责发送邮件\nSMTP: Only for sending'),
    (10, 2.5, 'POP3: 下载到本地（单设备）\nIMAP: 服务器同步（多设备）'),
]
for x, y, text in notes:
    ax.text(x, y, text, ha='center', va='center', fontsize=10, style='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#F8F9FA', edgecolor='gray', alpha=0.8))

# 流程编号
ax.text(3.5, 7, 'Step 1:\nCompose & Send\n撰写并发送', ha='center', va='center', fontsize=9, color='#E74C3C')
ax.text(7.8, 7, 'Step 2:\nRelay to Receiver\n转发到收件服务器', ha='center', va='center', fontsize=9, color='#E74C3C')
ax.text(12.5, 3.2, 'Step 3:\nDownload/Sync\n下载或同步', ha='center', va='center', fontsize=9, color='#3498DB')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 演示：模拟 SMTP 邮件发送会话
# ============================================================

print("=" * 60)
print("模拟 SMTP 会话（邮件发送过程）")
print("=" * 60)
print()
print("SMTP 是一个基于文本的协议，下面展示客户端和服务器之间的对话：")
print()

smtp_dialog = [
    ('S', '220 smtp.example.com ESMTP ready', '服务器准备就绪'),
    ('C', 'HELO client.example.com', '客户端打招呼'),
    ('S', '250 Hello client.example.com', '服务器确认'),
    ('C', 'MAIL FROM: <alice@example.com>', '指定发件人'),
    ('S', '250 OK', '服务器接受发件人'),
    ('C', 'RCPT TO: <bob@example.com>', '指定收件人'),
    ('S', '250 OK', '服务器接受收件人'),
    ('C', 'DATA', '开始发送邮件正文'),
    ('S', '354 Start mail input', '服务器准备接收'),
    ('C', 'From: alice@example.com', '邮件头 - 发件人'),
    ('C', 'To: bob@example.com', '邮件头 - 收件人'),
    ('C', 'Subject: Hello from A-Level CS', '邮件头 - 主题'),
    ('C', '', '空行（分隔头部和正文）'),
    ('C', 'Hi Bob,', '邮件正文'),
    ('C', 'This is a test email.', '邮件正文'),
    ('C', '.', '单独一个点表示正文结束'),
    ('S', '250 OK: Message queued', '邮件已接收并排队'),
    ('C', 'QUIT', '客户端断开连接'),
    ('S', '221 Bye', '服务器关闭连接'),
]

for role, message, description in smtp_dialog:
    prefix = '→ Client:' if role == 'C' else '← Server:'
    color_marker = '  ' if role == 'C' else '  '
    print(f"  {prefix:12} {message:<40} # {description}")

print("\n" + "=" * 60)
print("关键 SMTP 命令总结:")
print("=" * 60)
commands = [
    ('HELO/EHLO', '向服务器打招呼（EHLO 支持扩展功能）'),
    ('MAIL FROM', '指定发件人地址'),
    ('RCPT TO', '指定收件人地址'),
    ('DATA', '开始传输邮件内容'),
    ('QUIT', '结束会话'),
]
for cmd, desc in commands:
    print(f"  {cmd:<12} {desc}")

In [ ]:
# ============================================================
# 可视化：客户端-服务器通信总览
# ============================================================

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('Client-Server Communication Protocols Summary\n客户端-服务器通信协议总览',
             fontsize=16, fontweight='bold', pad=15)

# 客户端
rect_c = FancyBboxPatch((0.3, 4.5), 2.5, 3, boxstyle="round,pad=0.2",
                         facecolor='#AED6F1', edgecolor='black', linewidth=2)
ax.add_patch(rect_c)
ax.text(1.55, 6, 'Client\n客户端', ha='center', va='center', fontsize=14, fontweight='bold')

# 各种服务器
servers = [
    (10, 10.5, 'Web Server\nHTTP(S): 80/443', '#ABEBC6'),
    (10, 8.5, 'DNS Server\nUDP: 53', '#F9E79F'),
    (10, 6.5, 'Mail Server (Send)\nSMTP: 25/587', '#F5B7B1'),
    (10, 4.5, 'Mail Server (Recv)\nPOP3:110 / IMAP:143', '#D2B4DE'),
    (10, 2.5, 'FTP Server\nFTP: 20/21', '#FADBD8'),
    (10, 0.8, 'SSH Server\nSSH: 22', '#D5DBDB'),
]

for x, y, label, color in servers:
    rect = FancyBboxPatch((x - 1.8, y - 0.6), 5.5, 1.2,
                          boxstyle="round,pad=0.1", facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 0.95, y, label, ha='center', va='center', fontsize=10, fontweight='bold')
    # 连接线
    ax.annotate('', xy=(x - 1.8, y), xytext=(2.8, 6),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.2, alpha=0.5,
                               connectionstyle='arc3,rad=0.05'))

# 标注 Internet
ax.text(6, 6, 'Internet\n互联网', ha='center', va='center', fontsize=13,
        fontweight='bold', color='#E74C3C',
        bbox=dict(boxstyle='circle,pad=0.5', facecolor='#FDEBD0', edgecolor='#E74C3C', linewidth=2))

plt.tight_layout()
plt.show()

---

## 7. 综合回顾 (Summary)

### 本节考试要点清单

| 主题 | 关键知识 |
|------|----------|
| **DNS** | 将域名转换为 IP；分层结构（Root → TLD → Authoritative）；缓存机制 |
| **DNS 查询类型** | Recursive（递归）= 解析器全程代劳；Iterative（迭代）= 逐步引导 |
| **HTTP** | 无状态、请求-响应模式；方法：GET, POST, PUT, DELETE |
| **HTTP 状态码** | 200=OK, 404=Not Found, 500=Server Error |
| **HTTPS** | HTTP + TLS 加密；端口 443 |
| **TLS 握手** | 非对称加密交换密钥 → 对称加密传输数据 |
| **Web 三件套** | HTML=结构, CSS=样式, JavaScript=行为 |
| **REST API** | 使用 HTTP 方法操作资源；CRUD 对应 POST/GET/PUT/DELETE |
| **JSON** | 轻量级数据交换格式；键值对、数组、嵌套 |
| **SMTP** | 发送邮件；端口 25/587 |
| **POP3** | 下载邮件到本地；端口 110/995 |
| **IMAP** | 同步邮件（多设备）；端口 143/993 |

---

## 8. 练习题 (Practice Exercises)

### 练习 1：基础概念题

**Q1.** 描述 DNS 解析 `www.cambridge.org` 的完整过程（假设没有缓存）。说明每一步涉及的服务器类型。

**Q2.** 解释 HTTP 协议中 GET 和 POST 方法的区别。各举一个使用场景。

**Q3.** 什么是 HTTPS？它与 HTTP 的主要区别是什么？TLS 握手过程中为什么同时使用对称加密和非对称加密？

**Q4.** 比较 POP3 和 IMAP 协议。在什么场景下应该选择 IMAP 而不是 POP3？

---

### 练习 2：HTTP 状态码

**Q5.** 解释以下 HTTP 状态码的含义：
- (a) 200
- (b) 301
- (c) 403
- (d) 404
- (e) 500

---

### 练习 3：JSON 题

In [ ]:
# ============================================================
# 练习 3a: 解析 JSON 数据
# ============================================================

# 以下是一个学校 API 返回的 JSON 数据
school_json = '''
{
    "school": "Cambridge International School",
    "year": 2026,
    "classes": [
        {
            "name": "A2 Computer Science",
            "teacher": "Mr. Zhang",
            "students": [
                {"name": "Alice", "score": 85, "grade": "A"},
                {"name": "Bob", "score": 72, "grade": "B"},
                {"name": "Charlie", "score": 91, "grade": "A*"}
            ]
        },
        {
            "name": "AS Mathematics",
            "teacher": "Ms. Li",
            "students": [
                {"name": "David", "score": 68, "grade": "C"},
                {"name": "Eve", "score": 95, "grade": "A*"}
            ]
        }
    ]
}
'''

# 任务：解析上面的 JSON 并完成以下操作
# 1. 打印学校名称
# 2. 打印所有课程的名称和老师
# 3. 找出所有获得 "A*" 的学生姓名
# 4. 计算 A2 Computer Science 班级的平均分

# 在下面写你的代码：
data = json.loads(school_json)

# 1. 学校名称
# print(f"学校: {data[???]}")

# 2. 课程和老师
# for cls in data[???]:
#     print(f"  {cls[???]} - {cls[???]}")

# 3. A* 学生
# ...

# 4. 平均分
# ...

In [ ]:
# ============================================================
# 练习 3b: 构造 HTTP 请求
# ============================================================

def build_http_request(method, path, host, body=None):
    """
    构造一个完整的 HTTP 请求字符串。
    
    参数:
        method: HTTP 方法 (GET, POST, etc.)
        path: 请求路径 (如 /api/users)
        host: 主机名 (如 www.example.com)
        body: 请求正文 (字符串, 可选)
    
    返回:
        完整的 HTTP 请求字符串
    
    示例:
        build_http_request('GET', '/index.html', 'www.example.com')
        应返回:
        "GET /index.html HTTP/1.1\r\nHost: www.example.com\r\n..."
    """
    # 在下面写你的代码
    # 1. 构造请求行: "METHOD PATH HTTP/1.1\r\n"
    # 2. 添加 Host header
    # 3. 如果有 body，添加 Content-Length 和 Content-Type
    # 4. 空行分隔头部和正文
    # 5. 添加 body（如果有）
    pass

# 测试
# request = build_http_request('GET', '/api/students', 'school.example.com')
# print(request)

---

### 练习 4：Past Paper 风格题

**Q6.** (6 marks) A student visits `https://www.school.edu/results` using a web browser.

(a) Explain the role of DNS in this process. [2]

(b) Describe the steps involved in the TLS handshake that occurs before the web page is loaded. [4]

---

**Q7.** (5 marks) A company uses email for internal communication.

(a) Name the protocol used to **send** an email and the protocol used to **receive** emails. [2]

(b) Compare POP3 and IMAP. Explain which protocol would be more suitable for an employee who uses both a laptop and a smartphone to check emails. [3]

---

**Q8.** (6 marks) A web application provides a RESTful API for managing a library.

(a) Explain what is meant by a RESTful API. [2]

(b) The API provides the following endpoint: `GET /api/books/42`
   - What does this request do? [1]
   - What HTTP status code would be returned if the book exists? [1]
   - What HTTP status code would be returned if the book does not exist? [1]

(c) Write the HTTP request method and URL that would be used to create a new book record. [1]

---

> **本章完成！** 你已经学习了互联网技术的核心知识。建议复习时结合 `01_网络协议栈.ipynb` 的内容，构建完整的知识体系。